In [ ]:
import torch
from unsloth import FastLanguageModel
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from pathlib import Path

persist_directory = "/workspaces/Arch_PC_Assistent/embed_model"
embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-small-en-v1.5")
vectorstore = Chroma(persist_directory="./arch_vector_db", 
                     embedding_function=embeddings,
                     persist_directory = persist_directory)


path_model_v1 = Path("/workspaces/Arch_PC_Assistent/outputs/SFT/arch_assistant_lora_1").resolve()
max_seq_length = 4096 
model_name = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit" 

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = path_model_v1,
    max_seq_length = max_seq_length,
    dtype = None,
    load_in_4bit = True,
)
FastLanguageModel.for_inference(model)


test_frage = "My Waybar is crashing on startup. How can I debug or fix it?"

docs = vectorstore.similarity_search(test_frage, k=3)
rag_context = "\n\n".join([f"--- Chunk {i+1} ({d.metadata.get('topic')}) ---\n{d.page_content}" for i, d in enumerate(docs)])

system_prompt = f"""You are ArchAgent, an expert Arch Linux assistant. 
Use ONLY the following context to answer the user's question. If the answer is not in the context, say so.
Your output MUST follow this format:
<think>
[Your reasoning here]
</think>
<answer>
[Your final concise answer here]
</answer>
CONTEXT:
{rag_context}"""

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": test_frage}
]
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer([prompt], return_tensors="pt").to("cuda")


outputs = model.generate(
    **inputs,
    max_new_tokens=512,
    use_cache=True,
    do_sample=True,      # WICHTIG für Varianz!
    temperature=0.7,     # Macht das Modell kreativer in der Lösungsfindung
    top_p=0.9,           # Schneidet komplett unlogische Wörter ab
    num_return_sequences=6 # Erzeugt 6 Antworten auf einmal
)

# Ergebnisse anzeigen
decoded_outputs = tokenizer.batch_decode(outputs[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True)

for i, answer in enumerate(decoded_outputs):
    print(f"{"="*40}")
    print(f"🌟 ANTWORT {i+1} 🌟")
    print(f"{"="*40}")
    print(answer)
    print("\n")